# JetBot Button + Joystick Teleoperation

The stock teleop notebook wires the **left joystick axis directly to the motors** (`traitlets.dlink`). Because that's a raw, continuous, linear mapping, any small stick drift or noise translates instantly into motor movement — which is what makes it feel oversensitive.

This notebook uses two complementary schemes instead:

| Control | Action |
|---|---|
| D-pad Up | Drive forward at current speed |
| D-pad Down | Drive backward at current speed |
| D-pad Left | Pivot turn left |
| D-pad Right | Pivot turn right |
| Left joystick (push/pull) | Left motor spins clockwise (push) / counter-clockwise (pull), at current speed |
| Right joystick (push/pull) | Right motor spins clockwise (push) / counter-clockwise (pull), at current speed |
| Right Bumper (RB) | Increase speed |
| Left Bumper (LB) | Decrease speed |
| X button | Emergency stop + reset speed |

The joysticks give you tank-drive style independent left/right wheel control, but — like the D-pad — they only ever command the shared, adjustable `speed` value rather than a value proportional to how far you push the stick. A small deadzone around center ignores stick drift/noise, so you get clean on/off motion instead of a jittery analog signal.

**Note:** the D-pad and joysticks both drive the motors, so if you use both at once, whichever input changes most recently wins. In practice, pick one scheme at a time.

## 1. Initialize the robot and controller widget

Go to the gamepad tester website http://hardwaretester.com/gamepad and note the index number; if needed, edit the “ … widgets.Controller(index= )“ line in the cell.

In [ ]:
import traitlets
from jetbot import Robot
import ipywidgets.widgets as widgets
from IPython.display import display

robot = Robot()

controller = widgets.Controller(index=0)  # change index if you have more than one gamepad connected
display(controller)

Before continuing: press a couple of buttons on the physical controller. The widget above needs at least one input event before `controller.buttons` / `controller.axes` populate. If it stays blank, make sure the browser has recognized the gamepad.

## 2. (Optional) Find your button and axis indices

Button and axis numbering isn't identical across every controller. Run this cell, then press buttons and move both joysticks on your controller one at a time, and watch the output to confirm which index is which (D-pad, bumpers, A button, left stick Y, right stick Y). Skip this step if you already know your controller's mapping.

In [ ]:
button_output = widgets.Output()
display(button_output)

def on_button_change(change, index):
    if change['new'] > 0:
        with button_output:
            print(f"Button {index} pressed (value={change['new']:.2f})")

def on_axis_change(change, index):
    with button_output:
        print(f"Axis {index} value={change['new']:.2f}")

for i, button in enumerate(controller.buttons):
    button.observe(lambda change, index=i: on_button_change(change, index), names='value')

for i, axis in enumerate(controller.axes):
    axis.observe(lambda change, index=i: on_axis_change(change, index), names='value')

print(f"Watching {len(controller.buttons)} buttons and {len(controller.axes)} axes.")
print("Press each button and move each stick fully up/down to identify indices.")

In [ ]:
# Run this once you're done identifying indices, so the diagnostic
# handlers above don't stick around and clutter behavior later.
for button in controller.buttons:
    button.unobserve_all()
for axis in controller.axes:
    axis.unobserve_all()

## 3. Configure the mapping and speeds

Edit the indices below to match what you found in step 2. Button values are typical for an Xbox-style gamepad in Chrome; axis 1 (left stick Y) and axis 3 (right stick Y) are the standard positions. Confirm against your own results — in the browser Gamepad API, pushing a stick *up/forward* usually reads as a **negative** value, so that's the assumption baked into the joystick handler below. If your forward push reads positive instead, flip the sign in `JOY_FORWARD_SIGN`.

In [ ]:
# --- Button mapping: adjust to match YOUR controller ---
BTN_FORWARD  = 12   # D-pad Up
BTN_BACKWARD = 13   # D-pad Down
BTN_LEFT     = 14   # D-pad Left
BTN_RIGHT    = 15   # D-pad Right
BTN_SPEED_UP = 5    # Right bumper (RB)
BTN_SPEED_DN = 4    # Left bumper (LB)
BTN_ESTOP    = 2    # X button

# --- Axis mapping: adjust to match YOUR controller ---
AXIS_LEFT_Y  = 1    # Left stick, vertical
AXIS_RIGHT_Y = 3    # Right stick, vertical
JOY_DEADZONE = 0.2  # ignore stick movement smaller than this (drift/noise)
JOY_FORWARD_SIGN = 1  # -1 because pushing up usually reads as a negative axis value;
                        # flip to 1 if your diagnostic showed the opposite

# --- Speed settings ---
DEFAULT_SPEED = 0.4
DEFAULT_TURN_SPEED = 0.35
SPEED_STEP = 0.1
SPEED_MIN = 0.1
SPEED_MAX = 1.0

speed = DEFAULT_SPEED
turn_speed = DEFAULT_TURN_SPEED

## 4. Wire the D-pad and joysticks to robot motion

In [ ]:
DIRECTION_BUTTONS = [BTN_FORWARD, BTN_BACKWARD, BTN_LEFT, BTN_RIGHT]

def update_motion(change=None):
    """Reads the current state of the four direction buttons and drives
    the robot accordingly. Called any time one of them is pressed or released."""
    up    = controller.buttons[BTN_FORWARD].value  > 0.5
    down  = controller.buttons[BTN_BACKWARD].value > 0.5
    left  = controller.buttons[BTN_LEFT].value     > 0.5
    right = controller.buttons[BTN_RIGHT].value    > 0.5

    if up and not down:
        robot.forward(speed)
    elif down and not up:
        robot.backward(speed)
    elif left and not right:
        robot.left(turn_speed)
    elif right and not left:
        robot.right(turn_speed)
    else:
        robot.stop()

def joystick_motor_value(axis_value):
    """Maps a raw stick position to a fixed +-speed with a center deadzone,
    rather than a value proportional to how far the stick is pushed."""
    signed_value = axis_value * JOY_FORWARD_SIGN
    if signed_value > JOY_DEADZONE:
        return speed        # pushed forward -> clockwise
    elif signed_value < -JOY_DEADZONE:
        return -speed       # pulled backward -> counter-clockwise
    else:
        return 0.0

def update_left_joystick(change):
    robot.left_motor.value = joystick_motor_value(change['new'])

def update_right_joystick(change):
    robot.right_motor.value = joystick_motor_value(change['new'])

def refresh_joysticks():
    """Re-applies the current stick positions, used after a speed change so
    a stick that's already held over responds to the new speed immediately."""
    robot.left_motor.value = joystick_motor_value(controller.axes[AXIS_LEFT_Y].value)
    robot.right_motor.value = joystick_motor_value(controller.axes[AXIS_RIGHT_Y].value)

def change_speed(change, delta):
    global speed, turn_speed
    if change['new'] > 0.5:  # only act on press, not release
        speed = round(max(SPEED_MIN, min(SPEED_MAX, speed + delta)), 2)
        turn_speed = round(max(SPEED_MIN, min(SPEED_MAX, turn_speed + delta)), 2)
        print(f"Speed: {speed:.2f}   Turn speed: {turn_speed:.2f}")
        update_motion()      # re-apply D-pad state if a direction is held
        refresh_joysticks()  # re-apply joystick state if a stick is held

def emergency_stop(change):
    global speed, turn_speed
    if change['new'] > 0.5:
        robot.stop()
        speed = DEFAULT_SPEED
        turn_speed = DEFAULT_TURN_SPEED
        print("EMERGENCY STOP - speed reset to defaults")

# Direction buttons
for i in DIRECTION_BUTTONS:
    controller.buttons[i].observe(update_motion, names='value')

# Joysticks
controller.axes[AXIS_LEFT_Y].observe(update_left_joystick, names='value')
controller.axes[AXIS_RIGHT_Y].observe(update_right_joystick, names='value')

# Speed buttons
controller.buttons[BTN_SPEED_UP].observe(lambda c: change_speed(c, SPEED_STEP), names='value')
controller.buttons[BTN_SPEED_DN].observe(lambda c: change_speed(c, -SPEED_STEP), names='value')

# Emergency stop
controller.buttons[BTN_ESTOP].observe(emergency_stop, names='value')

print("Controller wired up:")
print("  D-pad          -> drive / turn (both wheels together)")
print("  Left stick     -> left motor (push=clockwise, pull=counter-clockwise)")
print("  Right stick    -> right motor (push=clockwise, pull=counter-clockwise)")
print("  RB / LB        -> speed up / down")
print("  A              -> emergency stop")
print(f"Starting speed: {speed:.2f}   Starting turn speed: {turn_speed:.2f}")

## 5. Disconnect when finished

Run this before closing the notebook or restarting the kernel, so the robot doesn't keep responding to a controller that's no longer being watched intentionally.

In [ ]:
for i in DIRECTION_BUTTONS + [BTN_SPEED_UP, BTN_SPEED_DN, BTN_ESTOP]:
    controller.buttons[i].unobserve_all()
controller.axes[AXIS_LEFT_Y].unobserve_all()
controller.axes[AXIS_RIGHT_Y].unobserve_all()
robot.stop()
print("Disconnected. Robot stopped.")